This notebook implements the algorithm described in the following paper:

M. Friese, "Multitone signals with low crest factor," in _IEEE Transactions on Communications_, vol. 45, no. 10, pp. 1338-1344, Oct. 1997, doi: 10.1109/26.634697

In [ ]:
import os
import numpy as np
from scipy.fft import fft, fftfreq
from scipy.integrate import trapezoid

import sys
if '..' not in sys.path:
    sys.path = ['..'] + sys.path
from multitone import optimize_phases, multitone_opt, compute_amplitude_distribution, compute_fourier_coeffs

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
F0 = 10e-3
T = 1 / F0
Δω = ω0 = 2 * np.pi * F0
N = 50
print(f'F0 = {F0:g} Hz')
print(f'T = {T:g} s')
print(f'Number of tones: {N}')

In [ ]:
fname = f'../multitone_phases_F0_{F0:.4f}_Hz_N_tones_{N}.npz'
force = False
if os.path.isfile(fname) and not force:
    print('Loading phases from {}.'.format(os.path.basename(fname)))
    data = np.load(fname)
    phases = data['phases']
    CF = data['CF']
    S = data['S']
    dt = data['dt']
else:
    phases, CF, S, t, _ = optimize_phases(Δω, N_tones=N, N_samples=5001, N_iters=2000, N_reps=10)
    dt = t[1] - t[0]
CF_min = CF.min()

In [ ]:
print('Minimum crest factor for no. of tones = {}: {:g} ({:g} dB).'.format(N, CF_min, 20 * np.log10(CF_min)))
fig,ax = plt.subplots(2, 1, figsize=(6,4), sharex=True)
ax[0].plot(CF)
ax[1].plot(S)
ax[1].set_xlabel('Iteration')
ax[0].set_ylabel('CF')
ax[1].set_ylabel('S')
sns.despine()
fig.tight_layout()

In [ ]:
N_T = 2
tt = np.r_[0 : N_T * T + dt / 2 : dt]
u = multitone_opt(tt, phases=phases, dw=Δω, w0=ω0)
n, x = compute_amplitude_distribution(u, bins='fd')

uf = fft(u)
ups = 2 / u.size * np.abs(uf[:u.size // 2])
freq = fftfreq(u.size, dt)[:u.size//2]

F = np.array([(ω0 + i * Δω) / (2 * np.pi) for i in range(N)])
coeffs = compute_fourier_coeffs(F, tt, u, phases)

In [ ]:
fig,ax = plt.subplots(3, 1, figsize=(6,5))
ax[0].plot(tt, u, lw=0.75)
yl = ax[0].get_ylim()
ax[0].vlines(np.arange(N_T + 1) * T, yl[0], yl[1], color='tab:red', lw=0.75, ls=':')
ax[0].set_xlabel('Time [s]')
ax[0].set_ylabel('u')

ax[1].plot(x, n, 'k', lw=0.75)
ax[1].set_ylim([0, 1])
ax[1].set_xlabel('Amplitude')
ax[1].set_ylabel(r'$F_u(a)$')

coeff = 1 if F0 >= 0.1 else 1e3

ax[2].vlines(freq * coeff, 0 * ups, ups, lw=0.5)
ax[2].plot(freq * coeff, ups, 'ks', markerfacecolor='w', markersize=4, markeredgewidth=1)
ax[2].plot(F * coeff, np.abs(coeffs), 'o', color='tab:red', markerfacecolor='tab:red', markersize=2)
ax[2].set_xlim([(ω0 - Δω) / (2 * np.pi) * coeff, (ω0 + N * Δω) / (2 * np.pi) * coeff])
ax[2].set_xlabel('Frequency [{}Hz]'.format('m' if coeff == 1e3 else ''))
ax[2].set_ylabel('|FFT(u)|')

sns.despine()
fig.tight_layout()